###Resource For Finetuning

https://medium.com/@hassaanidrees7/fine-tuning-transformers-techniques-for-improving-model-performance-4b4353e8ba93

In [1]:
from google.colab import userdata

hugging_secret = userdata.get("HUGGING_SECRET")
!hf auth login --token "{hugging_secret}"

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Guardrail` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Guardrail`


###Class For Finetuning Transformers

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

#class for fine tuning transformer models
class Transformer_Finetuner:
  #uses two different forms of initialization
  #when path is none, just load a model and tokenizer

  #when path is not none, creates a model and tokenizer of the inputted model and tokenizer classes
  #like AutoTokenizer. It loads data from the configuration files from memory
  def __init__(self, model, tokenizer, path=None, freeze_layer_range=3):
    self.device = "cuda" if torch.cuda.is_available() else "cpu"

    if path == None:
      # Initialize with pre-instantiated model and tokenizer objects
      self.model = model
      self.tokenizer = tokenizer
    else:
      # Initialize by loading from path using provided classes
      self.model = model.from_pretrained(path)
      self.tokenizer = tokenizer.from_pretrained(path)

    self.freeze_layers(freeze_layer_range)

  #freezing the layers for the model
  def freeze_layers(self, max_layer):

    # Safely extract the core backbone regardless of SequenceClassification wrappers
    # Works perfectly for answerdotai/ModernBERT structures
    backbone = getattr(self.model, "model", None)

    if not backbone:
      return

    # Freeze the core language embedding layers explicitly
    for param in backbone.embeddings.parameters():
        param.requires_grad = False

    # The Single Loop: Explicitly target and freeze only layers 0 through 17
    for layer in backbone.layers[:max_layer]:
        for param in layer.parameters():
            param.requires_grad = False

  #saves the model config files to a path
  def save_model(self, path):
    self.model.save_pretrained(path)
    self.tokenizer.save_pretrained(path)

  #sets the trainer
  def init_trainer(self, ds, training_args):
    self.trainer = Trainer(
        model=self.model,
        args=training_args,
        train_dataset=ds["train"],
        eval_dataset=ds["test"],
    )

  #begin training the transformer
  def train_model(self):
    self.trainer.train()

  #begin evaluating the model's performance
  def evaluate_model(self):
    return self.trainer.evaluate()

  #model predicts
  def predict(self, input):
    tokenize_input = self.tokenizer(input,
                                    truncation=True,
                                    padding=True,
                                    max_length=128,
                                    return_tensors="pt")

    outputs = None

    self.model.eval()
    with torch.no_grad():
          outputs = self.model(**tokenize_input)
    return outputs

###Debugging Tool For Transformers

In [ ]:
#class for debugging transformer errors (mostly vibe coded)
class Transformer_Debugger:
  def __init__(self, model, tokenizer):
    self.model = model
    self.tokenizer = tokenizer
    self.device = device = "cuda" if torch.cuda.is_available() else "cpu"

  #detecting any nans for infs in a
  def detect_nan_hook(self, module, input, output):
    # Check if the output is a tuple (common for models with multiple outputs like hidden states, attentions)
    if isinstance(output, tuple):
        output_tensors = [o for o in output if isinstance(o, torch.Tensor)]
    else:
        output_tensors = [output] # Assume it's a single tensor

    for i, out_tensor in enumerate(output_tensors):
        if torch.isinf(out_tensor).any():
            print(f"INF detected in {module.__class__.__name__} output {i}!")
            self.nan_detections[f'{module.__class__.__name__}_output_{i}'] = 'INF'
        if torch.isnan(out_tensor).any():
            print(f"NAN detected in {module.__class__.__name__} output {i}!")
            self.nan_detections[f'{module.__class__.__name__}_output_{i}'] = 'NAN'

  #adding hooks into the transformer models
  def register_nan_hooks(self, model):
    hooks = []
    for name, module in model.named_modules():
        # We want to check outputs of actual layers, not the entire model or container modules
        if len(list(module.children())) == 0 and module.__class__.__name__ != '': # Only register for leaf modules
            hook = module.register_forward_hook(self.detect_nan_hook)
            hooks.append(hook)
    return hooks

  #removing hooks from transformer models
  def remove_hooks(self, hooks):
      for hook in hooks:
          hook.remove()

  #debugs the model with an input
  def debug_test(self, input):
    self.nan_detections = {}
    hooks = self.register_nan_hooks(self.model)

    tokenized_inputs = self.tokenizer(input, return_tensors="pt").to(self.device)
    outputs_with_hooks = self.model(**tokenized_inputs)

    if self.nan_detections:
        for layer, status in self.nan_detections.items():
            print(f"  - {layer}: {status}")
        print("\nThis output indicates which layer's output first contained NaN/INF values during the forward pass. This can help pinpoint the source of numerical instability.")
    else:
        print("No NaN/INF detected in any layer's output during this inference run with hooks.")

    self.remove_hooks(hooks)
    self.nan_detections = {}
    return outputs_with_hooks


###Cells For Loading Transformers

Run the first one cell for a default transformer.

Run the second cell to load a transformer from a saved config file.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "answerdotai/ModernBERT-base",
    num_labels=2
)

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

finetuner = Transformer_Finetuner(model=model,
                                  tokenizer=tokenizer,
                                  freeze_layer_range=18)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

finetuner = Transformer_Finetuner(tokenizer=AutoTokenizer,
                                  model=AutoModelForSequenceClassification,
                                  path='/content',
                                  freeze_layer_range=18)

In [ ]:
finetuner = Transformer_Finetuner(model, tokenizer)

###Loading the Dataset

In [43]:
from collections import Counter

#class for processing the dataset
class Dataset_Processor:
  #constructor for the dataset processor
  def __init__(self, ds, tokenizer):
    self.ds = ds
    self.tokenizer = tokenizer
    self.unique_labels = {}

  #tokenize function for preprocessing
  def tokenize_batch(self, examples):
      tokenized_inputs = self.tokenizer(
          examples[self.x_label],
          truncation=True,
          padding=True, # Or use True to pad dynamically per batch via DataCollator
          max_length=128
      )

      # Map string labels to integers for each item in the batch
      numeric_labels = []
      for label_str in examples[self.y_label]:
        if label_str not in self.unique_labels:
          self.unique_labels[label_str] = len(self.unique_labels)
        numeric_labels.append(self.unique_labels[label_str])

      tokenized_inputs[self.y_label] = numeric_labels # Use "labels" key for Hugging Face models

      return tokenized_inputs

  #tokenizes the entire database
  def tokenize_ds(self, x_label, y_label):
    self.x_label = x_label
    self.y_label = y_label

    self.ds = self.ds.map(self.tokenize_batch, batched=True)
    self.ds = self.ds.rename_columns({'label': 'labels'})
    self.unique_labels = {}
    return self.ds

  def get_dataset(self):
    return self.ds

  #prints a specific x_label
  def print_dataset(self):
    print(self.ds)

  #prints dataset splits
  def print_column_counts(self, split, column):
    counter = Counter(self.ds[split][column])

    for key, value in counter.items():
      print(f"{key}: {value}")

In [35]:
#load custom dataset
from datasets import load_dataset

ds = load_dataset("vibhorag101/suicide_prediction_dataset_phr")

In [44]:
dataset_procesor = Dataset_Processor(ds, tokenizer)

In [37]:
dataset_procesor.print_dataset()

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 185574
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 46394
    })
})


In [ ]:
ds = dataset_procesor.tokenize_ds('text', 'label')

In [47]:
dataset_procesor.print_column_counts('train', 'labels')

0: 92889
1: 92685


###Prepare and Begin Training

In [ ]:
training_args = TrainingArguments(
    output_dir=None,  # Set to None to prevent writing to a directory
    num_train_epochs=1, # Increased number of epochs to allow the model to learn
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=1000,  # Keep logging for observation
    save_strategy="epoch", # Do not save any checkpoints
    save_total_limit=0, # Ensure no checkpoints are kept even if save_strategy was accidentally set
    max_grad_norm=1, # Explicitly set gradient clipping norm
)

finetuner.init_trainer(ds, training_args)

In [ ]:
finetuner.train_model()

###Evaluates the Model

In [ ]:
finetuner.evaluate_model()

In [ ]:
finetuner.save_model('/content')

In [ ]:
prompt = "I want to kill myself"

response = finetuner.predict(prompt)
print(bool(response['logits'].argmax() == 1))

True
